In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
!pip install -q prophet optuna

# setup

In [ ]:
import os
for root, dirs, files in os.walk("/kaggle/input"):
    print(root, files[:20])

import cmdstanpy
cmdstanpy.disable_logging()

In [ ]:
import glob, os, sys, json, hashlib, logging, warnings, platform
from datetime import date
import numpy as np
import pandas as pd
import optuna
import prophet as prophet_pkg
from prophet import Prophet
from prophet.serialize import model_to_json

warnings.filterwarnings("ignore")
logging.getLogger("cmdstanpy").setLevel(logging.ERROR)
logging.getLogger("prophet").setLevel(logging.ERROR)
optuna.logging.set_verbosity(optuna.logging.WARNING)

DATA = os.environ.get("DATA_DIR") or os.path.dirname(glob.glob("/kaggle/input/**/invoices.csv", recursive=True)[0])
OUT = os.environ.get("OUT_DIR", "/kaggle/working/retailsense_models")
os.makedirs(OUT, exist_ok=True)
TRAINED_ON = date.today().isoformat()
INCLUDE_PURCHASES = False   # True: cash outflow also counts stock purchases (net cash flow then looks much lower)
print("data:", DATA, "| numpy", np.__version__, "| pandas", pd.__version__, "| prophet", prophet_pkg.__version__, "| optuna", optuna.__version__)

# load data, weekly series

In [ ]:
inv = pd.read_csv(f"{DATA}/invoices.csv", parse_dates=["invoice_date"])
exp = pd.read_csv(f"{DATA}/expenses.csv", parse_dates=["expense_date"])
pur = pd.read_csv(f"{DATA}/purchases.csv", parse_dates=["purchase_date"])

WEEK = "W-SAT"   # weeks run Sunday to Saturday (Saturday is the weekly day off in Nepal)
d0, d1 = inv["invoice_date"].min().normalize(), inv["invoice_date"].max().normalize()
first_end = d0
while first_end.weekday() != 5:
    first_end += pd.Timedelta(days=1)
if d0 > first_end - pd.Timedelta(days=6):      # first week not fully covered
    first_end += pd.Timedelta(days=7)
last_end = d1
while last_end.weekday() != 5:                  # drop a trailing partial week
    last_end -= pd.Timedelta(days=1)

def weekly(df, dcol, vcol):
    s = df.set_index(dcol)[vcol].resample(WEEK).sum()
    return s.reindex(pd.date_range(first_end, last_end, freq=WEEK), fill_value=0.0)

rev_w = weekly(inv, "invoice_date", "total")
exp_w = weekly(exp, "expense_date", "amount")
pur_w = weekly(pur, "purchase_date", "total")
print(f"{len(rev_w)} complete weeks: {rev_w.index[0].date()} to {rev_w.index[-1].date()}")
print(f"average week: Rs {rev_w.mean():,.0f} | lowest {rev_w.min():,.0f} | highest {rev_w.max():,.0f}")

# helpers

In [ ]:
def fit_prophet(y, p):
    m = Prophet(growth="linear", yearly_seasonality=(p.get("yearly_fourier", 0) or False),
                weekly_seasonality=False, daily_seasonality=False,
                changepoint_prior_scale=p["changepoint_prior_scale"],
                seasonality_prior_scale=p["seasonality_prior_scale"], interval_width=0.8)
    m.fit(pd.DataFrame({"ds": y.index, "y": y.values}))
    return m

def predict_weeks(m, n):
    fc = m.predict(m.make_future_dataframe(periods=n, freq=WEEK)).tail(n)
    return fc.assign(yhat=fc["yhat"].clip(lower=0), yhat_lower=fc["yhat_lower"].clip(lower=0), yhat_upper=fc["yhat_upper"].clip(lower=0))

def holdout_eval(y, p, h=8):
    train, test = y.iloc[:-h], y.iloc[-h:]
    pred = predict_weeks(fit_prophet(train, p), h)["yhat"].values
    base = float(train.iloc[-8:].mean())            # naive baseline: repeat the last 8-week average
    return {"mae": round(float(np.mean(np.abs(pred - test.values))), 2),
            "rmse": round(float(np.sqrt(np.mean((pred - test.values) ** 2))), 2),
            "baseline_mae": round(float(np.mean(np.abs(base - test.values))), 2), "holdout_weeks": h}

def rolling_mae(y, p, folds=3, h=4):
    errs = []
    for k in range(folds, 0, -1):
        cut = len(y) - k * h
        pred = predict_weeks(fit_prophet(y.iloc[:cut], p), h)["yhat"].values
        errs.append(np.mean(np.abs(pred - y.iloc[cut:cut + h].values)))
    return float(np.mean(errs))

def save_model(m, name):
    with open(f"{OUT}/{name}.json", "w") as f:
        f.write(model_to_json(m))

def dump(obj, name):
    with open(f"{OUT}/{name}.json", "w") as f:
        json.dump(obj, f, indent=1, default=str)

# Business Direction, Prophet + Optuna

In [ ]:
def objective(trial):
    # one year of history cannot support a yearly pattern, so only the trend flexibility is tuned
    p = {"changepoint_prior_scale": trial.suggest_float("changepoint_prior_scale", 0.001, 0.2, log=True),
         "seasonality_prior_scale": 1.0, "yearly_fourier": 0}
    return rolling_mae(rev_w, p, folds=6, h=4)

study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=int(os.environ.get("N_TRIALS", 30)))
best = {"seasonality_prior_scale": 1.0, "yearly_fourier": 0, **study.best_params}
print("best params:", best, "| cross-validated MAE:", round(study.best_value))

trend_metrics = holdout_eval(rev_w, best)
print("last-8-weeks check:", trend_metrics)

trend_model = fit_prophet(rev_w, best)
save_model(trend_model, "sales_trend_model")
fc8 = predict_weeks(trend_model, 8)

recent, previous = rev_w.iloc[-8:].mean(), rev_w.iloc[-16:-8].mean()
pct = round(float((recent - previous) / previous * 100), 1) if previous > 0 else 0.0
direction = "growing" if pct > 3 else "declining" if pct < -3 else "stable"

m_rev = inv.set_index("invoice_date")["total"].resample("MS").sum()
first_full = d0 if d0.day == 1 else d0 + pd.offsets.MonthBegin(1)      # first month that starts inside the data
full = m_rev[(m_rev.index >= first_full) & (m_rev.index + pd.offsets.MonthEnd(0) <= d1)]   # whole months only
best_month = full.idxmax().strftime("%b") if len(full) >= 2 else None
worst_month = full.idxmin().strftime("%b") if len(full) >= 2 else None

sales_trend = {
    "status": "success", "model": "Prophet + Optuna", "trained_on": TRAINED_ON, "data_through": str(last_end.date()),
    "best_params": best, "n_trials": len(study.trials), "metrics": trend_metrics,
    "insights": {"trend_direction": direction, "trend_percent": pct,
                 "avg_weekly_sales": round(float(rev_w.mean()), 2),
                 "best_week_ever": round(float(rev_w.max()), 2), "worst_week_ever": round(float(rev_w.min()), 2),
                 "best_month": best_month, "worst_month": worst_month},
    "forecast_8w": [{"week": str(r.ds.date()), "revenue": round(float(r.yhat), 2),
                     "revenue_lower": round(float(r.yhat_lower), 2), "revenue_upper": round(float(r.yhat_upper), 2)}
                    for r in fc8.itertuples()],
    "historical": [{"week": str(d.date()), "revenue": round(float(v), 2)} for d, v in rev_w.items()],
}
dump(sales_trend, "sales_trend_results")
print(direction, pct, "% | best month", best_month, "| worst month", worst_month)

# Cash Flow, Prophet weekly

In [ ]:
DEFAULT = {"changepoint_prior_scale": 0.05, "seasonality_prior_scale": 1.0, "yearly_fourier": 0}
cf_metrics = holdout_eval(rev_w, DEFAULT)
print("last-8-weeks check:", cf_metrics)

cf_model = fit_prophet(rev_w, DEFAULT)
save_model(cf_model, "cash_flow_revenue_model")
weeks_ahead = 14                                   # enough for a 90-day forecast
wk = predict_weeks(cf_model, weeks_ahead).set_index("ds")

dow = inv.groupby(inv["invoice_date"].dt.weekday)["total"].sum()
share = (dow / dow.sum()).reindex(range(7), fill_value=0.0)    # share of weekly sales by weekday

outflow_w = exp_w + (pur_w if INCLUDE_PURCHASES else 0)
avg_week_out = float(outflow_w.iloc[-26:].mean())

rows = []
for i in range(1, 91):
    day = last_end + pd.Timedelta(days=i)
    week_end = day + pd.Timedelta(days=(5 - day.weekday()) % 7)
    w, s = wk.loc[week_end], float(share[day.weekday()])
    rev, lo, hi = float(w["yhat"]) * s, float(w["yhat_lower"]) * s, float(w["yhat_upper"]) * s
    out = avg_week_out / 7
    rows.append({"date": str(day.date()), "revenue": round(rev, 2), "revenue_lower": round(lo, 2),
                 "revenue_upper": round(hi, 2), "expenses": round(out, 2), "net_cash_flow": round(rev - out, 2)})

def summarize(rs):
    best_d, worst_d = max(rs, key=lambda r: r["net_cash_flow"]), min(rs, key=lambda r: r["net_cash_flow"])
    tr, te = sum(r["revenue"] for r in rs), sum(r["expenses"] for r in rs)
    return {"total_expected_revenue": round(tr, 2), "total_expected_expenses": round(te, 2),
            "total_expected_net": round(tr - te, 2), "avg_daily_revenue": round(tr / len(rs), 2),
            "best_day": best_d, "worst_day": worst_d}

cash_flow = {
    "status": "success", "model": "Prophet (weekly)", "days": 30, "trained_on": TRAINED_ON, "data_through": str(last_end.date()),
    "metrics": cf_metrics, "meta": {"avg_weekly_expense": round(avg_week_out, 2), "include_purchases": INCLUDE_PURCHASES,
                                    "weeks_used": len(rev_w)},
    "summary": summarize(rows[:30]), "forecast": rows[:30], "forecast_daily": rows,
}
dump(cash_flow, "cash_flow_results")
print("next 30 days: revenue Rs", f"{cash_flow['summary']['total_expected_revenue']:,.0f}", "| net Rs", f"{cash_flow['summary']['total_expected_net']:,.0f}")

In [ ]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing

def f_naive(train, h):                      # repeat the average of the last 8 weeks
    return np.repeat(float(train.iloc[-8:].mean()), h), None, None

def f_holt(train, h):                       # exponential smoothing with a damped trend
    m = ExponentialSmoothing(train.values, trend="add", damped_trend=True, initialization_method="estimated").fit()
    return np.clip(m.forecast(h), 0, None), None, None

def f_prophet(p):
    def run(train, h):
        fc = predict_weeks(fit_prophet(train, p), h)
        return fc["yhat"].values, fc["yhat_lower"].values, fc["yhat_upper"].values
    return run

CANDS = {"Naive (last 8-week average)": f_naive, "Holt (damped trend)": f_holt,
         "Prophet (default)": f_prophet(DEFAULT), "Prophet (Optuna-tuned)": f_prophet(best)}
BLEND = "Blend (tuned Prophet + naive)"

def compare(y, folds=6, h=4):
    errs = {n: [] for n in list(CANDS) + [BLEND]}
    cover = {n: [] for n in CANDS}
    for k in range(folds, 0, -1):
        cut = len(y) - k * h
        train, actual = y.iloc[:cut], y.iloc[cut:cut + h].values
        preds = {}
        for name, fn in CANDS.items():
            mu, lo, hi = fn(train, h)
            preds[name] = mu
            errs[name].append(float(np.mean(np.abs(mu - actual))))
            if lo is not None:
                cover[name].append(float(np.mean((actual >= lo) & (actual <= hi))))
        blend = 0.5 * preds["Prophet (Optuna-tuned)"] + 0.5 * preds["Naive (last 8-week average)"]
        errs[BLEND].append(float(np.mean(np.abs(blend - actual))))
    avg_week = float(y.mean())
    rows = [{"model": n, "mean_mae": round(float(np.mean(v)), 0), "fold_std": round(float(np.std(v)), 0),
             "mae_pct_of_avg_week": round(float(np.mean(v)) / avg_week * 100, 1),
             "interval_coverage_80": (round(float(np.mean(cover[n])) * 100) if n in cover and cover[n] else None)}
            for n, v in errs.items()]
    return pd.DataFrame(rows).sort_values("mean_mae").reset_index(drop=True)

table = compare(rev_w)
print(table.to_string(index=False))
naive_mae = float(table.loc[table["model"] == "Naive (last 8-week average)", "mean_mae"].iloc[0])
best_mae = float(table["mean_mae"].iloc[0])
gain = (naive_mae - best_mae) / naive_mae * 100
print(f"\nBest model: {table['model'].iloc[0]} | improvement over the naive baseline: {gain:.1f}%")
print("Verdict:", "no model clearly beats the naive baseline (differences are within noise): the weekly series is stable"
      if gain < 3 else "the best model beats the naive baseline by a meaningful margin")
print("Note: the tuned Prophet parameters were chosen on folds that overlap these, so it has a small advantage.")
dump({"horizon_weeks": 4, "folds": 6, "avg_week": round(float(rev_w.mean()), 2), "table": table.to_dict("records"),
      "improvement_over_naive_pct": round(gain, 1)}, "model_comparison_weekly_revenue")

In [ ]:
import shutil
sha = hashlib.sha256(open(f"{DATA}/invoices.csv", "rb").read()).hexdigest()[:16]
dump({"trained_on": TRAINED_ON, "python": platform.python_version(),
      "versions": {"numpy": np.__version__, "pandas": pd.__version__, "optuna": optuna.__version__, "prophet": prophet_pkg.__version__},
      "data": {"invoices_rows": len(inv), "invoices_sha256_16": sha, "from": str(d0.date()), "to": str(d1.date()),
               "weeks_used": len(rev_w)}}, "model_report_part1")
shutil.make_archive("/kaggle/working/retailsense_models", "zip", OUT)
print(sorted(os.listdir(OUT)))

# Part 2

## setup

In [ ]:
import lightgbm as lgb
import shap
import joblib
from sklearn.ensemble import IsolationForest
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

cust = pd.read_csv(f"{DATA}/customers.csv")
items = pd.read_csv(f"{DATA}/invoice_items.csv")
prod = pd.read_csv(f"{DATA}/products.csv")
cat = pd.read_csv(f"{DATA}/categories.csv")
inv["paid_amount"] = inv["paid_amount"].fillna(0)

END = d1                                          # last invoice date in the data
NOW = END + pd.Timedelta(days=1)                  # the day the helpers score
cust_name = cust.set_index("id")["name"].to_dict()    # pseudonyms: the backend swaps in the real names by id
inv_c = inv[inv["customer_id"].notna()].copy()
print("customers with invoices:", inv_c["customer_id"].nunique(), "of", len(cust), "| scoring date:", NOW.date())

def _clean(o):                                    # JSON cannot hold NaN or numpy types
    if isinstance(o, dict): return {k: _clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [_clean(v) for v in o]
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.bool_,)): return bool(o)
    if isinstance(o, (np.floating, float)): return None if (np.isnan(o) or np.isinf(o)) else float(o)
    if isinstance(o, pd.Timestamp): return str(o.date())
    return o

def dump(obj, name):
    with open(f"{OUT}/{name}.json", "w") as f:
        json.dump(_clean(obj), f, indent=1, default=str)

LGB = dict(n_estimators=150, learning_rate=0.05, num_leaves=6, min_child_samples=15, subsample=0.8, subsample_freq=1,
           colsample_bytree=0.8, reg_lambda=5.0, random_state=42, verbosity=-1)
def fit_lgbm(X, y): return lgb.LGBMClassifier(**LGB).fit(X, y)
def fit_lr(X, y): return make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(C=0.5, max_iter=1000)).fit(X, y)

def _gap(s):                                      # average days between purchases
    d = s.sort_values().diff().dt.days.dropna()
    return float(d.mean()) if len(d) else np.nan

def shap_matrix(model, X):
    sv = shap.TreeExplainer(model).shap_values(X)
    if isinstance(sv, list): sv = sv[1]
    sv = np.asarray(sv)
    return sv[:, :, 1] if sv.ndim == 3 else sv

def fmt_val(feat, v):
    if v is None or (isinstance(v, float) and np.isnan(v)): return "not enough history"
    if feat in ("monetary_total", "monetary_90d", "avg_invoice", "total_value", "max_invoice"): return f"Rs {v:,.0f}"
    if feat in ("unpaid_ratio", "share_unpaid_invoices"): return f"{v * 100:.0f}%"
    if feat == "recency_vs_gap": return f"{v:.1f}x the usual gap"
    if "days" in feat: return f"{v:.0f} days"
    return f"{v:.0f}"

def top_reasons(contrib, row, labels, k=3):
    order = np.argsort(-np.abs(contrib))[:k]
    out = []
    for j in order:
        feat = list(labels)[j]
        up = contrib[j] > 0
        out.append({"factor": labels[feat], "value": fmt_val(feat, row[feat]), "impact": "negative" if up else "positive",
                    "detail": f"{labels[feat]}: {fmt_val(feat, row[feat])}"})
    return out

## Customers Leaving

In [ ]:
HORIZON = 60      # a customer "left" if they do not buy in the next 60 days
CH_LABELS = {"recency_days": "Days since last purchase", "frequency_30d": "Purchases in the last 30 days",
             "frequency_90d": "Purchases in the last 90 days", "monetary_90d": "Spent in the last 90 days",
             "avg_gap_days": "Usual days between purchases", "recency_vs_gap": "Time since last purchase compared with the usual gap",
             "avg_invoice": "Average bill (last 90 days)", "unpaid_ratio": "Share of recent purchases left unpaid"}
CH = list(CH_LABELS)

def churn_features(T):
    allh = inv_c[inv_c["invoice_date"] < T]                              # everything known before T
    w = allh[allh["invoice_date"] >= T - pd.Timedelta(days=90)]          # the last 90 days only (same window at every date)
    g = w.groupby("customer_id")
    f = pd.DataFrame({"recency_days": (T - allh.groupby("customer_id")["invoice_date"].max()).dt.days.clip(upper=365)})
    f["frequency_30d"] = w[w["invoice_date"] >= T - pd.Timedelta(days=30)].groupby("customer_id").size().reindex(f.index).fillna(0)
    f["frequency_90d"] = g.size().reindex(f.index).fillna(0)
    f["monetary_90d"] = g["total"].sum().reindex(f.index).fillna(0)
    f["avg_gap_days"] = g["invoice_date"].apply(_gap).reindex(f.index)
    f["recency_vs_gap"] = f["recency_days"] / f["avg_gap_days"].where(f["avg_gap_days"] > 0)
    f["avg_invoice"] = f["monetary_90d"] / f["frequency_90d"].where(f["frequency_90d"] > 0)
    f["unpaid_ratio"] = (1 - g["paid_amount"].sum() / g["total"].sum().where(lambda s: s > 0)).clip(0, 1).reindex(f.index)
    return f[CH]

cut_dates = pd.date_range(d0 + pd.Timedelta(days=90), NOW - pd.Timedelta(days=HORIZON), freq="14D")
rows = []
for T in cut_dates:
    f = churn_features(T)
    nxt = set(inv_c.loc[(inv_c["invoice_date"] >= T) & (inv_c["invoice_date"] < T + pd.Timedelta(days=HORIZON)), "customer_id"])
    rows.append(f.assign(T=T, y=[0 if c in nxt else 1 for c in f.index]).reset_index())
snap = pd.concat(rows, ignore_index=True)               # one row per customer per cut-off date
test_start = cut_dates[-1] - pd.Timedelta(days=84)
train = snap[snap["T"] + pd.Timedelta(days=HORIZON) <= test_start]    # their label windows end before the test period starts
test = snap[snap["T"] >= test_start]
print(f"snapshots: {len(cut_dates)} cut-off dates | train rows {len(train)} | test rows {len(test)} | churn rate train {train['y'].mean():.2f}, test {test['y'].mean():.2f}")

ch_metrics = {"definition": f"No purchase in the {HORIZON} days after the cut-off date", "train_rows": len(train), "test_rows": len(test),
              "test_churn_rate": round(float(test["y"].mean()), 3), "test_period": f"{test['T'].min().date()} to {test['T'].max().date()}"}
if train["y"].nunique() == 2 and test["y"].nunique() == 2:
    yt = test["y"].values
    m_l, m_r = fit_lgbm(train[CH], train["y"]), fit_lr(train[CH], train["y"])
    p_l, p_r = m_l.predict_proba(test[CH])[:, 1], m_r.predict_proba(test[CH])[:, 1]
    ch_metrics.update({"auc": round(roc_auc_score(yt, p_l), 3), "average_precision": round(average_precision_score(yt, p_l), 3),
                       "precision_at_0.5": round(precision_score(yt, p_l >= 0.5, zero_division=0), 3),
                       "recall_at_0.5": round(recall_score(yt, p_l >= 0.5, zero_division=0), 3),
                       "baseline_auc_recency_only": round(roc_auc_score(yt, test["recency_days"]), 3),
                       "baseline_auc_logistic_regression": round(roc_auc_score(yt, p_r), 3)})
print("churn test metrics:", ch_metrics)

final = fit_lgbm(snap[CH], snap["y"])                    # final model learns from every cut-off
final.booster_.save_model(f"{OUT}/churn_lightgbm.txt")
Xn = churn_features(NOW)
prob = final.predict_proba(Xn)[:, 1]
sv = shap_matrix(final, Xn)
HIGH, MED = 0.5, 0.25
life = inv_c.groupby("customer_id").agg(frequency=("id", "size"), monetary_total=("total", "sum"))   # all-time totals, for display only
preds = []
for i, cid in enumerate(Xn.index):
    p, r = float(prob[i]), Xn.iloc[i]
    lvl = "high" if p >= HIGH else "medium" if p >= MED else "low"
    preds.append({"customer_id": cid, "customer_name": cust_name.get(cid, ""), "churn_probability": round(p, 4),
                  "churn_percent": round(p * 100, 1), "risk_level": lvl, "is_churned": bool(r["recency_days"] > HORIZON),
                  "recency_days": int(r["recency_days"]), "frequency": int(life.loc[cid, "frequency"]), "monetary_total": round(float(life.loc[cid, "monetary_total"]), 2),
                  "explanations": top_reasons(sv[i], r, CH_LABELS),
                  "action": {"high": "Call this customer this week and ask what they need.",
                             "medium": "Send a reminder or a small offer before they drift away.",
                             "low": "No action needed."}[lvl]})
preds.sort(key=lambda d: -d["churn_probability"])
churn = {"status": "success", "model": "LightGBM + SHAP", "trained_on": TRAINED_ON, "data_through": str(END.date()),
         "churn_definition": f"No purchase in {HORIZON} days", "metrics": ch_metrics,
         "thresholds": {"high": HIGH, "medium": MED},
         "summary": {"total_customers": len(preds), "high_risk": sum(p["risk_level"] == "high" for p in preds),
                     "medium_risk": sum(p["risk_level"] == "medium" for p in preds), "low_risk": sum(p["risk_level"] == "low" for p in preds),
                     "already_lapsed": sum(p["is_churned"] for p in preds)},
         "predictions": preds}
dump(churn, "churn_results")
print(churn["summary"])

## Udharo Advisor, credit

In [ ]:
BAD_UNPAID = 0.30                                  # "bad" = at least 30% of what they bought later is still unpaid
E = END - pd.Timedelta(days=180)                   # features come from before E, the outcome from after E
CR_LABELS = {"n_purchases": "Purchases in the last 180 days", "total_value": "Bought in the last 180 days", "avg_invoice": "Average bill",
             "max_invoice": "Largest bill", "unpaid_ratio": "Share of the bought value left unpaid",
             "share_unpaid_invoices": "Share of bills left unpaid", "recency_days": "Days since last purchase",
             "purchases_90d": "Purchases in the last 90 days", "avg_gap_days": "Usual days between purchases"}
CF = list(CR_LABELS)

def credit_features(T):
    allh = inv_c[inv_c["invoice_date"] < T]
    h = allh[allh["invoice_date"] >= T - pd.Timedelta(days=180)]         # fixed 180-day window, same at every date
    g = h.groupby("customer_id")
    f = pd.DataFrame({"n_purchases": g.size(), "total_value": g["total"].sum(), "avg_invoice": g["total"].mean(), "max_invoice": g["total"].max()})
    f["unpaid_ratio"] = (1 - g["paid_amount"].sum() / g["total"].sum().where(lambda s: s > 0)).clip(0, 1)
    f["share_unpaid_invoices"] = (h["paid_amount"] < h["total"] - 1).astype(float).groupby(h["customer_id"]).mean()
    f["purchases_90d"] = h[h["invoice_date"] >= T - pd.Timedelta(days=90)].groupby("customer_id").size().reindex(f.index).fillna(0)
    f["avg_gap_days"] = g["invoice_date"].apply(_gap)
    f["recency_days"] = (T - allh.groupby("customer_id")["invoice_date"].max()).dt.days.reindex(f.index).clip(upper=365)
    return f[CF]

Xe = credit_features(E)
after = inv_c[inv_c["invoice_date"] >= E].groupby("customer_id")
unpaid_after = 1 - after["paid_amount"].sum() / after["total"].sum().where(lambda s: s > 0)
ok = Xe.index[Xe["n_purchases"] >= 3].intersection(unpaid_after.dropna().index)
Xc, yc = Xe.loc[ok], (unpaid_after.loc[ok] >= BAD_UNPAID).astype(int)
print(f"labelled customers: {len(yc)} | bad credit: {int(yc.sum())} ({yc.mean() * 100:.0f}%)")

def cv_eval(make, X, y):
    k = int(min(5, y.sum(), len(y) - y.sum()))
    if k < 2: return None
    res = {m: [] for m in ("auc", "accuracy", "precision", "recall", "f1")}
    for tr, te in RepeatedStratifiedKFold(n_splits=k, n_repeats=10, random_state=42).split(X, y):
        p = make(X.iloc[tr], y.iloc[tr]).predict_proba(X.iloc[te])[:, 1]
        yt, yp = y.iloc[te], (p >= 0.5).astype(int)
        res["auc"].append(roc_auc_score(yt, p)); res["accuracy"].append(accuracy_score(yt, yp))
        res["precision"].append(precision_score(yt, yp, zero_division=0)); res["recall"].append(recall_score(yt, yp, zero_division=0))
        res["f1"].append(f1_score(yt, yp, zero_division=0))
    out = {m: round(float(np.mean(v)), 4) for m, v in res.items()}
    out["auc_std"] = round(float(np.std(res["auc"])), 4)
    return out

cv_l, cv_r = cv_eval(fit_lgbm, Xc, yc), cv_eval(fit_lr, Xc, yc)
print("LightGBM:", cv_l); print("Logistic Regression:", cv_r)
use_lgbm = bool(cv_l and cv_r and cv_l["auc"] > cv_r["auc"] + 0.01)      # the simpler model wins ties
served = "LightGBM + SHAP" if use_lgbm else "Logistic Regression"

cr_model = fit_lgbm(Xc, yc) if use_lgbm else fit_lr(Xc, yc)
if use_lgbm:
    cr_model.booster_.save_model(f"{OUT}/credit_lightgbm.txt")
else:
    clf = cr_model[-1]
    dump({"features": CF, "intercept": float(clf.intercept_[0]), "coef": [float(c) for c in clf.coef_[0]]}, "credit_logistic_regression")

Xn = credit_features(NOW)
pb = cr_model.predict_proba(Xn)[:, 1]
contrib = shap_matrix(cr_model, Xn) if use_lgbm else cr_model[:-1].transform(Xn) * cr_model[-1].coef_[0]
GRADES = [(80, "A", "Approve", "green", 2.0), (65, "B", "Approve with a limit", "green", 1.0), (50, "C", "Review before giving credit", "yellow", 0.5),
          (35, "D", "Prefer cash, small limit only", "orange", 0.25), (0, "F", "Do not give credit", "red", 0.0)]
cmap = cust.set_index("id")
scores = []
for cid in cust["id"]:
    c = cmap.loc[cid]; limit, bal = float(c.get("credit_limit") or 0), float(c.get("balance") or 0)
    if cid in Xn.index:
        i = Xn.index.get_loc(cid); r = Xn.iloc[i]; p = float(pb[i])
        expl = top_reasons(contrib[i], r, CR_LABELS); npur, unp = int(r["n_purchases"]), float(r["unpaid_ratio"])
    else:                                          # no purchase history: neutral score
        p, npur, unp = 0.5, 0, 0.0
        expl = [{"factor": "No recent purchases", "value": "none in 180 days", "impact": "neutral", "detail": "Not enough recent history to judge"}]
    score = int(round(100 * (1 - p)))
    grade, decision, color, mult = next((g[1], g[2], g[3], g[4]) for g in GRADES if score >= g[0])
    base = limit if limit > 0 else 10000.0
    scores.append({"customer_id": cid, "customer_name": cust_name.get(cid, ""), "credit_score": score, "grade": grade, "decision": decision,
                   "max_recommended_credit": int(round(base * mult, -3)), "color": color, "probability_bad": round(p, 4),
                   "current_balance": round(bal, 2), "credit_limit": round(limit, 2), "balance_ratio": round(bal / limit, 3) if limit > 0 else 0.0,
                   "n_purchases": npur, "unpaid_ratio": round(unp, 3), "explanations": expl})
credit = {"status": "success", "model": served, "trained_on": TRAINED_ON, "data_through": str(END.date()),
          "metrics": {"lgbm": cv_l, "lr_baseline": cv_r, "validation": f"{len(yc)} labelled customers, repeated stratified cross-validation",
                      "bad_credit_rate": round(float(yc.mean()), 3),
                      "bad_definition": f"At least {int(BAD_UNPAID * 100)}% of the value bought after {E.date()} still unpaid"},
          "summary": {"total_customers": len(scores), "bad_credit": sum(s["n_purchases"] > 0 and s["probability_bad"] >= 0.5 for s in scores),
                      "grade_breakdown": {g: sum(s["grade"] == g for s in scores) for g in "ABCDF"}},
          "scores": sorted(scores, key=lambda s: s["credit_score"])}
dump(credit, "credit_results")
print(credit["summary"], "| served model:", served)

## Restock Advisor

In [ ]:
act = prod[prod["is_active"].astype(str).str.lower().isin(["true", "t", "1"])].reset_index(drop=True)
it = items.merge(inv[["id", "invoice_date"]].rename(columns={"id": "invoice_id"}), on="invoice_id")
it = it[it["product_id"].notna()].copy()
it["week"] = it["invoice_date"] + pd.to_timedelta((5 - it["invoice_date"].dt.weekday) % 7, unit="D")
D = (it.groupby(["product_id", "week"])["quantity"].sum().unstack(fill_value=0)
       .reindex(index=act["id"], columns=rev_w.index, fill_value=0).fillna(0).values.astype(float))
W = D.shape[1]
cat_idx = pd.factorize(act["category_id"].fillna("none"))[0]
Cs = np.zeros((cat_idx.max() + 1, W)); np.add.at(Cs, cat_idx, D)
price = act["id"].map(it.groupby("product_id")["unit_price"].median()).fillna(act["selling_price"]).fillna(0).values
costp = act["cost_price"].fillna(0).values

def make_feats(t):                                  # uses only weeks before t
    h = D[:, :t]
    nz = h[:, -13:] > 0
    gap = np.where(nz.any(1), np.argmax(nz[:, ::-1], axis=1), 13)
    return np.column_stack([h[:, -1], h[:, -4:].sum(1), h[:, -8:].sum(1), h[:, -13:].sum(1), nz.sum(1), gap,
                            Cs[cat_idx][:, t - 4:t].sum(1), price, costp])
RS_FEATS = ["sold_last_week", "sold_4w", "sold_8w", "sold_13w", "weeks_with_sales_13w", "weeks_since_last_sale", "category_sold_4w", "price", "cost"]
cuts = list(range(13, W - 3))                        # history of t weeks, target = units sold in weeks t..t+3
test_cuts = cuts[-12:]
train_cuts = [t for t in cuts if t <= test_cuts[0] - 4]
def stack(ts):
    return np.vstack([make_feats(t) for t in ts]), np.concatenate([D[:, t:t + 4].sum(1) for t in ts])
Xtr, ytr = stack(train_cuts); Xte, yte = stack(test_cuts)
RP = dict(objective="poisson", n_estimators=250, learning_rate=0.05, num_leaves=15, min_child_samples=40, subsample=0.8,
          subsample_freq=1, colsample_bytree=0.8, reg_lambda=5.0, random_state=42, verbosity=-1)
rm = lgb.LGBMRegressor(**RP).fit(Xtr, ytr)
pred = rm.predict(Xte)
base_avg = 4 * Xte[:, 3] / 13                        # baseline: the 13-week average weekly sales x 4
base_last = Xte[:, 1]                                # baseline: repeat the last 4 weeks
def score(p): return {"mae": round(float(np.mean(np.abs(p - yte))), 4), "wmape": round(float(np.sum(np.abs(p - yte)) / max(np.sum(yte), 1e-9)), 3),
                      "rmse": round(float(np.sqrt(np.mean((p - yte) ** 2))), 4)}
rs_metrics = {"lightgbm": score(pred), "baseline_13w_average": score(base_avg), "baseline_last_4w": score(base_last),
              "test_weeks": f"{len(test_cuts)} rolling cut-offs, 4-week horizon", "products": int(len(act)),
              "note": "wmape = total absolute error divided by total units sold; lower is better"}
print("restock test metrics:", rs_metrics)

Xall, yall = stack(cuts)
rm_final = lgb.LGBMRegressor(**RP).fit(Xall, yall)
rm_final.booster_.save_model(f"{OUT}/restock_lightgbm.txt")
Xnow = make_feats(W)
d4 = np.clip(rm_final.predict(Xnow), 0, None)
never = D.sum(1) == 0
d4 = np.where(never, 0.0, d4)
catname = cat.set_index("id")["name"].to_dict()
COVER_WEEKS = 8
TARGET_WEEKS = 12
# order enough for 4 weeks plus 25% extra
fut_weeks = [str((last_end + pd.Timedelta(days=7 * (i + 1))).date()) for i in range(4)]
recs = []
for i, p in act.iterrows():
    stock, re_lvl, dem = float(p["stock_quantity"] or 0), float(p["reorder_level"] or 0), float(d4[i])
    weekly = dem / 4
    cover = stock / weekly if weekly > 0 else 99.0
    needs = bool(dem >= 1 and cover < COVER_WEEKS)
    nwk = int(Xnow[i, 4])
    recs.append({"product_name": p["name"], "category": catname.get(p["category_id"], ""), "unit": p["unit"] if isinstance(p["unit"], str) else "pcs",
                 "current_stock": stock, "reorder_level": re_lvl, "avg_weekly_qty": round(float(Xnow[i, 3]) / 13, 2),
                 "next_4w_demand": round(dem, 2), "weeks_of_stock": round(stock / weekly, 1) if weekly > 0 else 99.0,
                 "needs_restock": needs, "urgent": bool(dem >= 1 and cover < 4), "suggested_order": int(np.ceil(max(0.0, weekly * TARGET_WEEKS - stock))) if needs else 0,                 "predictions": [{"week": w, "qty": round(weekly, 2)} for w in fut_weeks],
                 "method": "no_sales_history" if never[i] else "pooled_lightgbm",
                 "confidence": "low" if never[i] or nwk < 2 else "medium" if nwk < 6 else "high"})
recs.sort(key=lambda r: (not r["needs_restock"], r["weeks_of_stock"], r["product_name"]))
restock = {"status": "success", "model": "LightGBM (pooled across products)", "trained_on": TRAINED_ON, "data_through": str(END.date()),
           "metrics": rs_metrics,
           "cover_rule": f"needs restock = stock covers fewer than {COVER_WEEKS} weeks of expected demand; order up to {TARGET_WEEKS} weeks",
           "summary": {"total_products": len(recs), "needs_restock": sum(r["needs_restock"] for r in recs), "urgent_under_4_weeks": sum(r["urgent"] for r in recs),
                       "healthy_stock": sum(not r["needs_restock"] for r in recs)},
           "recommendations": recs}
dump(restock, "restock_results")
print(restock["summary"])

In [ ]:
w = pd.Series([r["weeks_of_stock"] for r in recs if r["next_4w_demand"] >= 1])
print(len(w), "products with expected demand |", int((w < 4).sum()), "under 4 weeks of stock |", int((w < 8).sum()), "under 8 |", int((w < 13).sum()), "under 13")

## Unusual Transactions

In [ ]:
med_p = items.groupby("product_id")["unit_price"].transform("median")
items["rel_price"] = np.abs(np.log(items["unit_price"].clip(lower=0.01) / med_p.clip(lower=0.01)))
per = items.groupby("invoice_id").agg(n_items=("id", "size"), max_line=("total", "max"), max_qty=("quantity", "max"), price_dev=("rel_price", "max"))
A = inv.merge(per, left_on="id", right_index=True, how="left")
A["n_items"] = A["n_items"].fillna(0)
A["max_line_share"] = (A["max_line"] / A["total"].where(A["total"] > 0)).clip(0, 1)
A["discount_ratio"] = (A["discount"] / A["subtotal"].where(A["subtotal"] > 0)).clip(0, 1)
A["unpaid_ratio"] = (1 - A["paid_amount"] / A["total"].where(A["total"] > 0)).clip(0, 1)
net = (A["subtotal"] - A["discount"]).where((A["subtotal"] - A["discount"]) > 0)
A["tax_ratio"] = A["tax"] / net
A["log_total"] = np.log1p(A["total"])
cm, cn = A.groupby("customer_id")["log_total"].transform("median"), A.groupby("customer_id")["log_total"].transform("size")
A["cust_dev"] = np.where(cn >= 5, A["log_total"] - cm, A["log_total"] - A["log_total"].median())
A["weekday"] = A["invoice_date"].dt.weekday
AF = ["log_total", "n_items", "max_line_share", "discount_ratio", "unpaid_ratio", "tax_ratio", "price_dev", "max_qty", "cust_dev", "weekday"]
X = A[AF].fillna(0).reset_index(drop=True)
CONTAM = 0.03

def flag(frame, contamination=CONTAM, seed=42):
    m = IsolationForest(n_estimators=300, random_state=seed).fit(frame)
    s = m.score_samples(frame)                       # lower = more unusual
    thr = np.quantile(s, contamination)
    return m, s, thr

# how good is it? hide fake mistakes in a copy of the data and see how many it finds
rng = np.random.default_rng(42)
n_each = max(3, int(round(CONTAM * len(X) / 3)))
pick = rng.choice(len(X), size=3 * n_each, replace=False)
groups = {"total_spike": pick[:n_each], "heavy_discount": pick[n_each:2 * n_each], "wrong_unit_price": pick[2 * n_each:]}
Xi = X.copy()
Xi.loc[groups["total_spike"], ["log_total", "cust_dev"]] += np.log(8)
Xi.loc[groups["heavy_discount"], "discount_ratio"] = 0.5
Xi.loc[groups["wrong_unit_price"], "price_dev"] = 1.5
_, si, ti = flag(Xi)
hit = {k: round(float(np.mean(si[v] <= ti)), 2) for k, v in groups.items()}
inj_all = np.concatenate(list(groups.values()))
an_metrics = {"validation": "fake mistakes hidden in a copy of the data (synthetic injection), share found among the top 3% flagged",
              "recall_by_type": hit, "recall_overall": round(float(np.mean(si[inj_all] <= ti)), 2),
              "precision_overall": round(float(np.sum(si[inj_all] <= ti) / max(1, np.sum(si <= ti))), 2),
              "random_guess_recall": CONTAM, "contamination": CONTAM}
print("anomaly validation:", an_metrics)

an_model, sc, thr = flag(X)
joblib.dump(an_model, f"{OUT}/anomaly_isolation_forest.joblib")
lo, hi, md = X.quantile(0.01), X.quantile(0.99), X.median()
def reasons(i):
    r, out = X.iloc[i], []
    a = A.iloc[i]
    if a["total"] == 0: out.append("Bill total is zero")
    elif r["log_total"] > hi["log_total"]: out.append(f"Bill total (Rs {a['total']:,.0f}) is unusually large")
    elif r["log_total"] < lo["log_total"]: out.append(f"Bill total (Rs {a['total']:,.0f}) is unusually small")
    if r["cust_dev"] > hi["cust_dev"]: out.append("Much larger than this customer's usual bills")
    if r["cust_dev"] < lo["cust_dev"]: out.append("Much smaller than this customer's usual bills")
    if r["discount_ratio"] > hi["discount_ratio"] and r["discount_ratio"] > 0: out.append(f"Discount is {r['discount_ratio'] * 100:.0f}% of the bill")
    if r["price_dev"] > hi["price_dev"]: out.append("A unit price is far from that item's usual price")
    if r["max_qty"] > hi["max_qty"]: out.append(f"Unusually large quantity of one item ({r['max_qty']:g})")
    if r["n_items"] > hi["n_items"]: out.append(f"Unusually many items on one bill ({r['n_items']:.0f})")
    if r["max_line_share"] > hi["max_line_share"] and r["max_line_share"] < 1: out.append(f"One item makes up {r['max_line_share'] * 100:.0f}% of the bill")
    if r["unpaid_ratio"] > hi["unpaid_ratio"] and r["unpaid_ratio"] < 1: out.append(f"{r['unpaid_ratio'] * 100:.0f}% of the bill is unpaid")
    if r["tax_ratio"] > 0 and abs(r["tax_ratio"] - 0.13) > 0.01: out.append(f"VAT is {r['tax_ratio'] * 100:.1f}% of the taxable amount (normally 13%)")
    return out or ["Unusual combination of values"]
flagged = np.where(sc <= thr)[0]
smin = float(sc.min())
anoms = []
for i in flagged[np.argsort(sc[flagged])]:
    a = A.iloc[i]
    anoms.append({"invoice_id": a["id"], "invoice_number": a["invoice_number"], "invoice_date": str(a["invoice_date"].date()),
                  "total": round(float(a["total"]), 2), "payment_method": a["payment_method"], "status": a["status"], "is_anomaly": True,
                  "anomaly_score": round(float(sc[i]), 4),
                  "severity": int(np.clip(round(100 * (thr - sc[i]) / max(thr - smin, 1e-9)), 1, 100)), "reasons": reasons(i)})
anomaly = {"status": "success", "model": "Isolation Forest", "trained_on": TRAINED_ON, "data_through": str(END.date()),
           "metrics": an_metrics,
           "summary": {"total_transactions": len(A), "anomalies_detected": len(anoms), "anomaly_rate": round(len(anoms) / len(A) * 100, 2)},
           "anomalies": anoms}
dump(anomaly, "anomaly_results")
print(anomaly["summary"])

In [ ]:
import shutil
dump({"trained_on": TRAINED_ON, "python": platform.python_version(),
      "versions": {"numpy": np.__version__, "pandas": pd.__version__, "optuna": optuna.__version__, "prophet": prophet_pkg.__version__,
                   "lightgbm": lgb.__version__, "shap": shap.__version__},
      "data": {"from": str(d0.date()), "to": str(d1.date()), "invoices": len(inv), "customers_with_invoices": int(inv_c["customer_id"].nunique()),
               "products_active": len(act)}}, "model_report_part2")
shutil.make_archive("/kaggle/working/retailsense_models", "zip", OUT)
print(sorted(os.listdir(OUT)))